<strong>E-Commerce Customer Return Prediction System</strong>

In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score, f1_score

from sklearn.model_selection import train_test_split

In [2]:
from sklearn.preprocessing import MinMaxScaler

In [3]:
df = pd.read_csv(r'C:\ML_OPS Project\database\customer_details.csv')

In [4]:
df_original = pd.read_csv(r'C:\ML_OPS Project\database\customer_details.csv')

In [5]:
df.head()

,customer_age,product_price,discount_percent,product_rating,past_purchase_count,past_return_rate,delivery_delay_days,session_length_minutes,num_product_views,device_type,product_category,shipping_method,payment_method,used_coupon,returned,order_id
0,25,11.666137,43.559601,1.933533,8,0.331408,-0.089637,3.273029,26,tablet,sports,express,paypal,1,0,75381
1,76,31.378967,66.526039,0.690237,9,0.105203,3.639277,130.513123,8,mobile,electronics,standard,credit_card,1,0,65569
2,40,25.460603,20.762374,2.347620,14,0.236683,0.294060,92.780917,-2,desktop,toys,express,credit_card,0,0,163473
3,62,13.517170,25.669260,2.033596,9,0.410038,1.683333,103.004177,7,mobile,clothing,express,apple_pay,0,0,90518
4,51,75.771311,15.527927,4.907556,9,0.123565,-0.153691,100.752658,6,mobile,electronics,standard,credit_card,1,0,138866


In [6]:
df = df.drop('order_id', axis=1)
df = df.drop('delivery_delay_days', axis=1)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   customer_age            200000 non-null  int64  
 1   product_price           200000 non-null  float64
 2   discount_percent        200000 non-null  float64
 3   product_rating          200000 non-null  float64
 4   past_purchase_count     200000 non-null  int64  
 5   past_return_rate        200000 non-null  float64
 6   session_length_minutes  200000 non-null  float64
 7   num_product_views       200000 non-null  int64  
 8   device_type             200000 non-null  object 
 9   product_category        200000 non-null  object 
 10  shipping_method         200000 non-null  object 
 11  payment_method          200000 non-null  object 
 12  used_coupon             200000 non-null  int64  
 13  returned                200000 non-null  int64  
dtypes: float64(5), int64

In [8]:
df.describe()

,customer_age,product_price,discount_percent,product_rating,past_purchase_count,past_return_rate,session_length_minutes,num_product_views,used_coupon,returned
count,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.00000,200000.000000,200000.000000
mean,56.479545,37.984234,42.872056,2.812805,9.620210,0.215114,77.406138,14.65445,0.615400,0.474595
std,17.952930,45.626883,21.510490,1.219874,3.139979,0.137162,36.394216,12.65940,0.486502,0.499355
min,10.000000,-183.292092,-7.540832,0.445342,1.000000,-0.060878,-11.010479,-5.00000,0.000000,0.000000
25%,44.000000,6.876686,25.848785,1.789263,8.000000,0.118326,48.995295,3.00000,0.000000,0.000000
50%,58.000000,24.461142,42.945197,2.818731,9.000000,0.191392,81.562252,13.00000,1.000000,0.000000
75%,72.000000,53.355820,62.642248,3.810169,10.000000,0.282027,108.209400,25.00000,1.000000,1.000000
max,87.000000,855.078645,78.791547,5.452690,26.000000,0.866420,137.652911,44.00000,1.000000,1.000000


In [9]:
df.corr(numeric_only=True)['returned']

customer_age             -0.011384
product_price             0.034326
discount_percent          0.011751
product_rating            0.004555
past_purchase_count      -0.023705
past_return_rate          0.018014
session_length_minutes   -0.017329
num_product_views        -0.018842
used_coupon               0.054950
returned                  1.000000
Name: returned, dtype: float64

In [10]:
df = pd.get_dummies(df, columns=['device_type', 'product_category', 'shipping_method', 'payment_method'], drop_first=False)
df.corr(numeric_only=True)['returned']


customer_age                   -0.011384
product_price                   0.034326
discount_percent                0.011751
product_rating                  0.004555
past_purchase_count            -0.023705
past_return_rate                0.018014
session_length_minutes         -0.017329
num_product_views              -0.018842
used_coupon                     0.054950
returned                        1.000000
device_type_desktop             0.050691
device_type_mobile             -0.051532
device_type_tablet              0.009513
product_category_beauty         0.021575
product_category_clothing       0.041231
product_category_electronics   -0.053820
product_category_home          -0.025470
product_category_sports        -0.017340
product_category_toys           0.032153
shipping_method_express         0.069313
shipping_method_same_day       -0.018450
shipping_method_standard       -0.054034
payment_method_apple_pay        0.052609
payment_method_credit_card      0.009213
payment_method_d

In [11]:
df.lt(0).sum()

customer_age                        0
product_price                   27285
discount_percent                 1686
product_rating                      0
past_purchase_count                 0
past_return_rate                 2220
session_length_minutes           1487
num_product_views               21443
used_coupon                         0
returned                            0
device_type_desktop                 0
device_type_mobile                  0
device_type_tablet                  0
product_category_beauty             0
product_category_clothing           0
product_category_electronics        0
product_category_home               0
product_category_sports             0
product_category_toys               0
shipping_method_express             0
shipping_method_same_day            0
shipping_method_standard            0
payment_method_apple_pay            0
payment_method_credit_card          0
payment_method_debit_card           0
payment_method_paypal               0
dtype: int64

In [12]:
print("product_price: ", df['product_price'].max(), df['product_price'].min(),
"\ndiscount_percent: ", df['discount_percent'].max(), df['discount_percent'].min(),
"\npast_return_rate: ", df['past_return_rate'].max(), df['past_return_rate'].min(),
"\nsession_length_minutes: ", df['session_length_minutes'].max(), df['session_length_minutes'].min(),
"\nnum_product_views: ", df['num_product_views'].max(), df['num_product_views'].min())

product_price:  855.0786452162006 -183.2920921688182 
discount_percent:  78.79154678672955 -7.540832249618946 
past_return_rate:  0.8664200331772883 -0.0608784165936474 
session_length_minutes:  137.65291067255592 -11.010478695730718 
num_product_views:  44 -5


In [13]:
cols = [
    'product_price',
    'discount_percent',
    'past_return_rate',
    'session_length_minutes',
    'num_product_views'
]

for col in cols:
    df.loc[df[col] < 0, col] = np.nan

for col in cols:
    df[col] = df[col].fillna(df[col].median())

In [14]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('returned', axis=1), df['returned'], test_size=0.2, random_state=42)

In [15]:
scaler = MinMaxScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [16]:
linear_model = LogisticRegression()
linear_model.fit(X_train, y_train)

lr_pred = linear_model.predict(X_test)

print("Logistic regression Accuracy on test set:", accuracy_score(y_test, lr_pred))
print("Logistic regression confusion matrix set:\n", confusion_matrix(y_test, lr_pred))
print("Logistic regression recall score on test set:", recall_score(y_test, lr_pred))
print("Logistic regression precision score on test set:", precision_score(y_test, lr_pred))
print("Logistic regression F1 score on test set:", f1_score(y_test, lr_pred))

Logistic regression Accuracy on test set: 0.564675
Logistic regression confusion matrix set:
 [[14705  6278]
 [11135  7882]]
Logistic regression recall score on test set: 0.41447126255455646
Logistic regression precision score on test set: 0.5566384180790961
Logistic regression F1 score on test set: 0.4751484462127377


In [17]:
random_forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
random_forest_model.fit(X_train, y_train)

rf_test_pred = random_forest_model.predict(X_test)

print("Random Forest test accuracy:", accuracy_score(y_test, rf_test_pred))
print("Random Forest confusion matrix:\n", confusion_matrix(y_test, rf_test_pred))
print("Random Forest recall score on test set:", recall_score(y_test, rf_test_pred))
print("Random Forest precision score on test set:", precision_score(y_test, rf_test_pred))
print("Random Forest F1 score on test set:", f1_score(y_test, rf_test_pred))

Random Forest test accuracy: 0.548725
Random Forest confusion matrix:
 [[13773  7210]
 [10841  8176]]
Random Forest recall score on test set: 0.4299311142661829
Random Forest precision score on test set: 0.5313921747042766
Random Forest F1 score on test set: 0.4753073859837805


In [18]:
lr_prob = linear_model.predict_proba(X_test)
rf_prob = random_forest_model.predict_proba(X_test)

In [ ]:
feature_columns = df.drop(columns=["returned"]).columns.tolist()

categorical_columns = [
    "device_type",
    "product_category",
    "shipping_method",
    "payment_method"
]

categories = {}

for col in categorical_columns:
    categories[col] = sorted(
        df_original[col].dropna().astype(str).unique().tolist()
    )

models = {
    "logistic_regression": linear_model,
    "random_forest": random_forest_model,
    "scaler": scaler,
    "feature_columns": feature_columns,
    "categories": categories
}

with open("customer_return_models.pkl", "wb") as file:
    pickle.dump(models, file)

print("Models and categories saved successfully!")
print(categories)

Models and categories saved successfully!
{'device_type': ['desktop', 'mobile', 'tablet'], 'product_category': ['beauty', 'clothing', 'electronics', 'home', 'sports', 'toys'], 'shipping_method': ['express', 'same_day', 'standard'], 'payment_method': ['apple_pay', 'credit_card', 'debit_card', 'paypal']}
